# 02 · 分组聚合  GroupBy & Aggregation

> **能力标签**：SH4（数据处理与分析）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解 **split-apply-combine** 模式：按键分组 → 对每组计算 → 合并结果。
2. 用 `groupby(key)[col].mean()` / `sum()` / `count()` 等做单一聚合。
3. 用 `agg({col: [func, ...]})` 同时计算多种统计量。
4. 实现 `group_means(df, key, value)` —— W3-groupby 题包核心函数。

> 对应能力：**SH4**


## 概念讲解 Concepts

### Split-Apply-Combine

pandas 的 `groupby` 遵循三步模式：

```
原始 DataFrame
    │
    ▼ split（按 key 分组）
group A │ group B │ group C
    │
    ▼ apply（对每组运行函数）
结果 A  │ 结果 B  │ 结果 C
    │
    ▼ combine（拼合输出）
聚合后 DataFrame / Series
```

---

### 常用聚合方法

```python
grouped = df.groupby('group')

grouped['x1'].mean()                            # 各组均值
grouped['age'].agg(['mean', 'std', 'count'])     # 多种统计
grouped.agg({'x1': 'mean', 'age': 'max'})      # 按列指定聚合
```

---

### `as_index` 参数

| `as_index=True`（默认）| `as_index=False` |
|------------------------|------------------|
| 分组键成为 Index | 分组键留在列中（结果是普通 DataFrame）|

```python
df.groupby('group', as_index=False)['x1'].mean()
```

---

### `group_means` 函数签名（题包核心）

```python
def group_means(df, key, value):
    out = df.groupby(key, as_index=False)[value].mean()
    return out.sort_values(key).reset_index(drop=True)
```


## 示例 Worked Example

In [ ]:
import pandas as pd
import numpy as np

# 构造示例数据
rng = np.random.default_rng(42)
n = 100
df = pd.DataFrame({
    'record_id': range(n),
    'group':     rng.choice(['A', 'B'], n),
    'dept':       rng.choice(['A', 'B', 'C'], n),
    'x1':        rng.normal(27, 5, n).round(1),
    'age':        rng.integers(18, 80, n),
})

# 1. 基础 groupby
print('=== 各组 x1 均值 ===')
print(df.groupby('group')['x1'].mean().round(2))

# 2. 多种聚合
print('\n=== 各部门 x1 统计 ===')
print(df.groupby('dept')['x1'].agg(['mean', 'std', 'count']).round(2))

# 3. as_index=False 保留列
print('\n=== as_index=False 示例 ===')
result = df.groupby('dept', as_index=False)['age'].mean()
print(result)

# 4. 实现 group_means
def group_means(df, key, value):
    out = df.groupby(key, as_index=False)[value].mean()
    return out.sort_values(key).reset_index(drop=True)

print("\n=== group_means(df, 'group', 'x1') ===")
print(group_means(df, 'group', 'x1'))


## 动手 Exercise

在下面的 cell 中：

1. 用如下销售数据创建 DataFrame（或自行构造）。
2. 用 `groupby` 计算各 `category` 的 `revenue` 总和。
3. 用 `agg` 同时计算各 `category` 的 `revenue` 均值和标准差。
4. 调用 `group_means(df, 'category', 'revenue')`，观察输出。


In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
sales = pd.DataFrame({
    'category': rng.choice(['Electronics', 'Clothing', 'Food'], 50),
    'revenue':  rng.uniform(100, 2000, 50).round(2),
    'qty':      rng.integers(1, 20, 50),
})

# 1. 总和
print('=== 各品类营收总和 ===')
print(sales.groupby('category')['revenue'].sum().round(2))

# 2. 多种聚合
print('\n=== 均值 + 标准差 ===')
print(sales.groupby('category')['revenue'].agg(['mean', 'std']).round(2))

# 3. group_means
def group_means(df, key, value):
    out = df.groupby(key, as_index=False)[value].mean()
    return out.sort_values(key).reset_index(drop=True)

print('\n=== group_means ===')
print(group_means(sales, 'category', 'revenue'))


## 延伸阅读 Further Reading

1. **pandas 官方文档 — GroupBy**: <https://pandas.pydata.org/docs/user_guide/groupby.html>
2. **Split-Apply-Combine 论文**（Hadley Wickham, 2011）: <https://doi.org/10.18637/jss.v040.i01>
3. **《Python for Data Analysis》第 10 章** — Wes McKinney
4. **pandas `agg` 参考**: <https://pandas.pydata.org/docs/reference/groupby.html>


## 关联练习 Related Assignments

本课对应 **W3-groupby** 题包，核心函数为 `group_means`：

```bash
python check.py 01-groupby
```

> 能力标签：**SH4** · threshold ≥ 0.7
